# 11 — Map a query arm onto the scVI atlas (papermill-parameterized)

Uses scArches-style `load_query_data` to fine-tune new batch embeddings
while freezing the reference encoder. No prototype mechanism (unlike scPoli).

In [ ]:
from pathlib import Path
import sys
import time
import json
import numpy as np

sys.path.insert(0, str(Path.cwd().parent))
from src.paths import DATA_OUT, MODEL_OUT
import anndata as ad
from scvi.model import SCVI

In [ ]:
# --- PARAMETERS (overridden by papermill) ---
QUERY_SOURCE = "source_5"
T_ARM = "tplus"
REFERENCE_ATLAS = "scenario_5_full"   # "excl_{source}" or "scenario_5_full"
N_EPOCHS = 200
# --- END PARAMETERS ---

## Load atlas + query

In [ ]:
atlas_dir = MODEL_OUT / f"scvi_atlas_{REFERENCE_ATLAS}"
query = ad.read_h5ad(DATA_OUT / "query_arms" / f"{QUERY_SOURCE}_{T_ARM}.h5ad")
print(f"atlas: {atlas_dir}, query: {query.shape}")

# Apply the same non-negative shift used when training the reference atlas.
X_min = json.loads((atlas_dir / "X_min.json").read_text())["X_min"]
query.X = query.X - X_min
print(f"shifted query X by {X_min:.4f}")

## scArches load_query_data + fine-tune

In [ ]:
model_query = SCVI.load_query_data(
    adata=query,
    reference_model=str(atlas_dir),
    inplace_subset_query_vars=True,
)

t0 = time.perf_counter()
model_query.train(
    max_epochs=N_EPOCHS,
    plan_kwargs={"weight_decay": 0.0},
    early_stopping=True,
    early_stopping_monitor="elbo_validation",
)
elapsed = time.perf_counter() - t0
print(f"mapping took {elapsed:.1f}s")

In [ ]:
query.obsm["X_scvi_mapped"] = model_query.get_latent_representation()

## Persist mapped query + query model

In [ ]:
out_dir = DATA_OUT / "mapped"
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / f"{QUERY_SOURCE}_{T_ARM}_scvi.h5ad"
query.write_h5ad(out_path)
(out_dir / f"{QUERY_SOURCE}_{T_ARM}_scvi_timings.json").write_text(
    json.dumps({"mapping_seconds": elapsed, "n_epochs": N_EPOCHS})
)
print(f"wrote {out_path}")

model_save_path = out_dir / f"{QUERY_SOURCE}_{T_ARM}_scvi_model"
model_query.save(str(model_save_path), overwrite=True)
print(f"saved query model to {model_save_path}")